# Hypothesis Testing and Confidence Intervals Tutorial #
This notebook teaches us the core tools of statistical inference: hypothesis tests, confidence intervals, and power analysis.<br>
<br>
We use simulated data so that the true answer is known and we can check our methods against it.<br>
<br>
The notebook covers:
  - The logic of hypothesis testing and p-values
  - One-sample, two-sample, and paired t-tests
  - Confidence intervals, and what they do and do not mean
  - Bootstrap confidence intervals for skewed data
  - Power analysis and sample size for t-tests
  - Type I and Type II errors
  - Multiple testing corrections
<br>

## Theory ##

Statistical inference answers a question about a population using a sample. We never see the whole population, so every answer carries uncertainty. Hypothesis tests and confidence intervals are two ways of describing that uncertainty.

**Hypothesis testing**

1. Write a **null hypothesis** $H_0$, usually "no effect" or "no difference"
2. Write an **alternative hypothesis** $H_1$, the effect we are looking for
3. Assume $H_0$ is true and ask how surprising the data would be
4. The **p-value** is the probability of a result at least as extreme as the one observed, if $H_0$ is true
5. If the p-value is below the significance level $\alpha$ (usually 0.05), reject $H_0$

A p-value is **not** the probability that $H_0$ is true. It is a statement about the data given $H_0$, not about $H_0$ given the data.

**t-statistic**

When the population standard deviation is unknown, which is almost always, we estimate it with the sample standard deviation and use the t-distribution:

$$
t = \frac{\bar{x} - \mu_0}{s / \sqrt{n}}
$$

The t-distribution has heavier tails than the normal distribution. This makes it wider for small samples and close to the normal distribution as $n$ grows.

**Confidence intervals**

A 95% confidence interval for the mean is

$$
\bar{x} \pm t_{0.975,\,n-1} \cdot \frac{s}{\sqrt{n}}
$$

The correct reading is about the **method**: if we repeated the sampling many times and built an interval each time, about 95% of those intervals would contain the true mean. It does not mean there is a 95% chance the true mean is in this one interval.

**Errors and power**
  - **Type I error:** rejecting $H_0$ when it is true. Its rate is $\alpha$.
  - **Type II error:** failing to reject $H_0$ when a real effect exists. Its rate is $\beta$.
  - **Power** $= 1 - \beta$: the probability of detecting a real effect of a given size

**Real-world use**

These methods appear almost everywhere decisions are made from data:
  - **Manufacturing:** a quality engineer checks whether a filling line's average volume matches the label, and whether a process change shifted the average.
  - **Medicine:** a clinical trial compares average blood pressure between a drug group and a placebo group. The confidence interval shows the plausible size of the benefit, which regulators and doctors care about as much as the p-value.
  - **Engineering:** a team compares average page load times before and after a server upgrade, with a confidence interval that says how much faster the site is likely to be.
  - **Machine learning:** comparing two models on the same cross-validation folds is a paired test. Reporting a confidence interval for the score difference shows whether an improvement is real or within noise.
  - **Policy and science:** the confidence interval often matters more than the p-value. A tiny but significant effect with a wide interval is weak evidence, while a large effect with a narrow interval is strong evidence.

Because many decisions depend on these tools, the most common errors are not computational. They are misreading the p-value, running underpowered studies, and testing many things until something looks significant.
<br>

### Setup ###

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

%matplotlib inline

rng = np.random.default_rng(7)

### One-Sample t-Test ###
A one-sample t-test asks whether the mean of one group equals a known target.<br>
<br>
**Real-world example:** a bottling line is supposed to fill each bottle to 500 ml. We sample 40 bottles and test whether the average has drifted from the target.<br>
<br>
In practice this is used for quality control, checking calibration, and verifying that a measured average meets a specification.

In [ ]:
# Target fill volume in ml
target = 500

# Simulate 40 bottles from a line that is slightly underfilling
fill = rng.normal(loc=497, scale=6, size=40)

print(f"Sample mean: {fill.mean():.1f} ml")
print(f"Sample std:  {fill.std(ddof=1):.1f} ml")

In [ ]:
t_stat, p_value = stats.ttest_1samp(fill, popmean=target)

print(f"t statistic: {t_stat:.3f}")
print(f"p-value:     {p_value:.4f}")

In [ ]:
# 95% confidence interval for the mean, using the t-distribution
n = len(fill)
se = fill.std(ddof=1) / np.sqrt(n)
ci_low, ci_high = stats.t.interval(0.95, df=n - 1, loc=fill.mean(), scale=se)

print(f"95% CI for the mean: [{ci_low:.1f}, {ci_high:.1f}] ml")
print(f"Target of 500 ml inside the interval: {ci_low <= target <= ci_high}")

The test rejects the target mean, and the interval shows how far below 500 ml the line probably is. A manager can use the interval to decide whether the underfill is large enough to act on, not just whether it is statistically different.

### Two-Sample t-Test (Welch) ###
A two-sample t-test compares the means of two independent groups.<br>
<br>
**Real-world example:** a team moves to a new server configuration and wants to know whether average page load time changed. The two groups are different users or requests, so they are independent.<br>
<br>
We use **Welch's test** by default. It does not assume the two groups have equal variances, and it is the safer choice in practice. Equal variances are rarely guaranteed.

In [ ]:
# Page load times in seconds for two server configurations
old_server = rng.normal(loc=2.4, scale=0.8, size=60)
new_server = rng.normal(loc=2.1, scale=0.6, size=55)

t_stat, p_value = stats.ttest_ind(new_server, old_server, equal_var=False)

print(f"Old mean: {old_server.mean():.2f} s")
print(f"New mean: {new_server.mean():.2f} s")
print(f"t statistic: {t_stat:.3f}")
print(f"p-value:     {p_value:.4f}")

In [ ]:
# 95% confidence interval for the difference in means (new minus old)
diff = new_server.mean() - old_server.mean()

v_new = new_server.var(ddof=1) / len(new_server)
v_old = old_server.var(ddof=1) / len(old_server)
se = np.sqrt(v_new + v_old)

# Welch-Satterthwaite approximation for the degrees of freedom
df = (v_new + v_old) ** 2 / (
    v_new ** 2 / (len(new_server) - 1) + v_old ** 2 / (len(old_server) - 1)
)
ci_low, ci_high = stats.t.interval(0.95, df=df, loc=diff, scale=se)

print(f"Difference (new - old): {diff:+.2f} s")
print(f"95% CI for the difference: [{ci_low:+.2f}, {ci_high:+.2f}] s")

Notice that the interval is for the **difference**, which is the quantity a decision depends on. A negative interval means the new server is probably faster, and the interval shows by how much.<br>
<br>
Real load times are often right-skewed. For data like that, the bootstrap section below is a better option.

### Paired t-Test ###
A paired test compares two measurements taken on the same unit, such as the same user before and after a change, or the same patient on two treatments.<br>
<br>
**Real-world example:** two machine learning models are evaluated on the same 10 cross-validation folds. Each fold has its own difficulty, so both models score high on easy folds and low on hard ones. Pairing removes that shared variation.

In [ ]:
# Accuracy of two models on the same 10 cross-validation folds
# Some folds are harder than others, and both models find the same folds hard
fold_difficulty = rng.normal(0, 0.03, 10)
model_a = 0.85 + fold_difficulty + rng.normal(0, 0.005, 10)
model_b = model_a + 0.01 + rng.normal(0, 0.005, 10)

print(f"Mean accuracy A: {model_a.mean():.4f}")
print(f"Mean accuracy B: {model_b.mean():.4f}")
print(f"Mean difference (B - A): {(model_b - model_a).mean():+.4f}")

In [ ]:
t_paired, p_paired = stats.ttest_rel(model_b, model_a)
t_unpaired, p_unpaired = stats.ttest_ind(model_b, model_a, equal_var=False)

print(f"Paired t-test p-value:   {p_paired:.4f}")
print(f"Unpaired t-test p-value: {p_unpaired:.4f}")

The paired test gives a much smaller p-value from the same data. The unpaired test treats the fold-to-fold variation as noise and loses the signal. Whenever the same units appear in both groups, pairing is the right choice.

### Confidence Intervals in Practice ###
A confidence interval is a statement about a procedure. Here we check that claim directly. We draw many samples from a known population, build a 95% interval from each one, and count how often the interval contains the true mean.

In [ ]:
true_mean = 10
sims = 5000
covered = 0

for _ in range(sims):
    sample = rng.normal(true_mean, 3, size=25)
    se = sample.std(ddof=1) / np.sqrt(len(sample))
    low, high = stats.t.interval(0.95, df=len(sample) - 1, loc=sample.mean(), scale=se)
    covered += low <= true_mean <= high

print(f"Share of 95% CIs containing the true mean: {covered / sims:.1%}")

The coverage is close to 95%, which is the guarantee the method provides. Any single interval either contains the true value or does not. That is why the correct reading is about the method, and why a 95% interval is not a 95% probability for one specific range.

### Bootstrap Confidence Intervals ###
The t-based interval assumes the sampling distribution of the mean is approximately normal. That assumption is weak for large samples, but it can fail for small or heavily skewed data.<br>
<br>
The **bootstrap** avoids the assumption. We resample the observed data with replacement many times, recompute the statistic each time, and use the spread of those values as the uncertainty.<br>
<br>
**Real-world example:** API response times are right-skewed, with a long tail of slow requests. The mean is pulled up by those outliers, so the median is often the more useful summary. The bootstrap works for any statistic, including the median, which has no simple t-based formula.<br>
<br>
Bootstrapping is common in A/B testing for revenue metrics, which are skewed by a few large purchases, and in model evaluation for metrics like F1 or AUC.

In [ ]:
# Response times in milliseconds, right-skewed like real latency data
response_ms = rng.lognormal(mean=4.0, sigma=0.5, size=80)
median_ms = np.median(response_ms)

# Resample with replacement and recompute the median each time
boot_medians = np.array([
    np.median(rng.choice(response_ms, size=len(response_ms), replace=True))
    for _ in range(10000)
])
ci_low, ci_high = np.percentile(boot_medians, [2.5, 97.5])

print(f"Sample median: {median_ms:.1f} ms")
print(f"Sample mean:   {response_ms.mean():.1f} ms")
print(f"95% bootstrap CI for the median: [{ci_low:.1f}, {ci_high:.1f}] ms")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax.hist(boot_medians, bins=50, color="tab:blue", alpha=0.7)
ax.axvline(ci_low, linestyle="--", color="grey", label="2.5th percentile")
ax.axvline(ci_high, linestyle="--", color="grey", label="97.5th percentile")
ax.set_xlabel("Bootstrap median (ms)")
ax.set_ylabel("Count")
ax.set_title("Bootstrap distribution of the median")
ax.legend()
plt.show()

The histogram is the uncertainty estimate. The interval is the middle 95% of the bootstrap medians, and no normal distribution was assumed.

### Power Analysis for a t-Test ###
Power is the probability of detecting a real effect. Before collecting data, power analysis tells us how many observations we need.<br>
<br>
The effect size $d$ is the difference in means divided by the standard deviation. Common benchmarks are 0.2 (small), 0.5 (medium), and 0.8 (large), but the right size depends on the business or scientific question.<br>
<br>
**Real-world example:** a product team wants to detect a change in a metric that is hard to move. If they under-sample, they will run a test that cannot find the effect even when it exists, and waste the time spent running it.<br>
<br>
An approximate formula for users per group in a two-sided two-sample t-test is

$$
n \approx \frac{2\left(z_{1-\alpha/2} + z_{1-\beta}\right)^2}{d^2}
$$

The approximation is slightly low for small samples because the t-distribution is wider. Simulation confirms the answer.

In [ ]:
def power_ttest(n, d, alpha=0.05, sims=2000):
    """Simulated power of a two-sided two-sample t-test with n per group and effect size d."""
    hits = 0
    for _ in range(sims):
        a = rng.normal(0, 1, n)
        b = rng.normal(d, 1, n)
        _, p = stats.ttest_ind(b, a)
        hits += p < alpha
    return hits / sims


def n_per_group(d, alpha=0.05, power=0.8):
    """Approximate users per group needed for a given effect size."""
    z_alpha = stats.norm.ppf(1 - alpha / 2)
    z_beta = stats.norm.ppf(power)
    return int(np.ceil(2 * (z_alpha + z_beta) ** 2 / d ** 2))

for d in [0.2, 0.5, 0.8]:
    n = n_per_group(d)
    print(f"Effect size d = {d}: {n} per group, simulated power {power_ttest(n, d):.1%}")

In [ ]:
sizes = np.arange(10, 151, 10)
powers = [power_ttest(n, d=0.5) for n in sizes]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sizes, powers, marker="o")
ax.axhline(0.8, linestyle="--", color="grey", label="80% power")
ax.set_xlabel("Users per group")
ax.set_ylabel("Power")
ax.set_title("Power for a medium effect (d = 0.5)")
ax.legend()
plt.show()

### Type I Error Under Skewed Data ###
The significance level promises that a test rejects a true null hypothesis about 5% of the time. That promise can break with small samples from skewed data.<br>
<br>
**Real-world example:** wait times, transaction amounts, and session lengths are often skewed and have many small values. If a team runs many small tests on such data, more false positives appear than the nominal 5%.

In [ ]:
# Both groups come from the same skewed distribution, so any significant result is a false positive
sims = 5000
n_small = 15
hits = 0

for _ in range(sims):
    a = rng.exponential(scale=1.0, size=n_small)
    b = rng.exponential(scale=1.0, size=n_small)
    _, p = stats.ttest_ind(a, b, equal_var=False)
    hits += p < 0.05

print(f"False positive rate with n = {n_small} per group: {hits / sims:.1%}")

Here the false positive rate stayed close to 5%, which is reassuring for equal group sizes. The guarantee is approximate, though, and it is weaker when samples are small, groups are unequal, or the data are heavily skewed. In those cases, check the distribution, compare with a bootstrap interval, or collect more data before trusting a p-value.

### Multiple Testing ###
Each test has a 5% chance of a false positive. Run 20 tests and the chance that at least one is significant grows quickly, even when nothing is real.<br>
<br>
**Real-world example:** a genomics study tests thousands of genes against a disease. A lab that reports every gene with p < 0.05 would report many false leads. Corrections for multiple testing are standard in that field, and the same problem appears when a team checks many metrics or segments in an A/B test.<br>
<br>
Two common corrections:
  - **Bonferroni:** test each p-value against $\alpha / m$. It controls the chance of any false positive, but it is conservative.
  - **Benjamini-Hochberg:** controls the false discovery rate, the expected share of false positives among the discoveries. It has more power when many tests are real.

In [ ]:
# 100 features are tested against an outcome. Only the first 5 have a real effect.
n_tests = 100
n_obs = 50
true_effects = np.zeros(n_tests, dtype=bool)
true_effects[:5] = True

p_values = []
for is_real in true_effects:
    shift = 1.0 if is_real else 0.0
    group_a = rng.normal(0, 1, n_obs)
    group_b = rng.normal(shift, 1, n_obs)
    _, p = stats.ttest_ind(group_b, group_a)
    p_values.append(p)
p_values = np.array(p_values)

In [ ]:
def benjamini_hochberg(p, q=0.05):
    """Return a boolean array of discoveries that controls the false discovery rate at level q."""
    m = len(p)
    order = np.argsort(p)
    thresholds = q * np.arange(1, m + 1) / m
    passed = p[order] <= thresholds
    reject = np.zeros(m, dtype=bool)
    if passed.any():
        k = np.max(np.where(passed)[0])
        reject[order[: k + 1]] = True
    return reject

uncorrected = p_values < 0.05
bonferroni = p_values < 0.05 / n_tests
bh = benjamini_hochberg(p_values, q=0.05)

print(f"{'Method':<14}{'Discoveries':>12}{'Real effects found':>20}{'False discoveries':>19}")
for name, found in [("Uncorrected", uncorrected), ("Bonferroni", bonferroni), ("Benjamini-Hochberg", bh)]:
    true_found = np.sum(found & true_effects)
    print(f"{name:<14}{found.sum():>12}{true_found:>20}{found.sum() - true_found:>19}")

With 95 features that have no real effect, uncorrected testing is expected to produce about 5 false discoveries. This run produced 7, which is within the normal variation for a single set of tests. Bonferroni removes most of them, but it also misses some real effects. Benjamini-Hochberg sits between the two, and is the usual choice when screening many features or metrics.

## Summary ##
  - A p-value measures how surprising the data are if the null hypothesis is true. It is not the probability that the null is true.
  - A 95% confidence interval comes from a procedure that covers the true value 95% of the time, and it shows the size of the effect, not just whether it exists.
  - Use a paired test when the same units appear in both groups, and Welch's test when the groups are independent.
  - The bootstrap gives confidence intervals for skewed data and for statistics without simple formulas.
  - Plan sample size with power analysis before collecting data. Underpowered tests waste time.
  - Correct for multiple testing when many comparisons are made.

## Further Reading ##
  - *Introduction to Statistical Learning* (included in `Resources/`), which covers hypothesis tests and multiple testing in the context of machine learning
  - *Statistics Done Wrong* by Alex Reinhart, a short, practical guide to common statistical mistakes
  - SciPy statistics documentation: https://docs.scipy.org/doc/scipy/reference/stats.html